# 🔬 Model Pipeline: Meta Llama-3.1-8B-Instruct (4-bit QLoRA)
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization
**Model ID:** `meta-llama/Llama-3.1-8B-Instruct`
**Architecture:** CAUSAL (4-bit QLoRA)

---
### 🎯 Strict Uniform Benchmark Constraints (Scientifically Fair)
- **`MAX_SOURCE_LENGTH = 512` tokens**: Uniform input length across all 6 models.
- **`MAX_TARGET_LENGTH = 128` tokens**: Uniform output generation limit across all 6 models.

### 📊 The 3 Non-Redundant Benchmark Pillars Measured:
1. **Overall Quality**: ROUGE-L F1 (↑ Higher is Better) & Target F1 Score (↑)
2. **Hallucination Risk**: Ungrounded Token % (↓ Lower is Better) & Precision Source (% grounded in abstract)
3. **Inference Time**: Wall-Clock Latency per Sample (↓ Lower is Faster) & Throughput (tok/s ↑)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE_DIR = "/content/drive/MyDrive/JU_Benchmark_Project"
DATASET_DIR = os.path.join(DRIVE_BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(DRIVE_BASE_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

TRAIN_PATH = os.path.join(DATASET_DIR, "MixSub-SciHigh_train_FIRE.csv")
VAL_PATH = os.path.join(DATASET_DIR, "MixSub-SciHigh_val_FIRE.csv")
TEST_PATH = os.path.join(DATASET_DIR, "MixSub-SciHigh_test_FIRE(Masked).csv")

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

print("Drive setup complete.")
print("Dataset directory:", DATASET_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes rouge-score pandas huggingface_hub

import torch

assert torch.cuda.is_available(), "GPU not available. Enable GPU in Colab."

print("GPU:", torch.cuda.get_device_name(0))

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Compute dtype:", COMPUTE_DTYPE)

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import json
import time

from rouge_score import rouge_scorer

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

assert "Abstract" in train_df.columns
assert "Highlights" in train_df.columns
assert "Abstract" in val_df.columns
assert "Highlights" in val_df.columns

test_df = None

if os.path.exists(TEST_PATH):
    test_df = pd.read_csv(TEST_PATH)
    if "Abstract" not in test_df.columns:
        test_df = None

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df) if test_df is not None else "Unavailable")


def normalize_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()


def lexical_overlap(reference, prediction):
    ref_tokens = normalize_text(reference).split()
    pred_tokens = normalize_text(prediction).split()

    if not pred_tokens:
        return 0.0, 0.0, 0.0

    ref_set = set(ref_tokens)
    pred_set = set(pred_tokens)

    common = len(ref_set & pred_set)

    precision = common / len(pred_set) if pred_set else 0.0
    recall = common / len(ref_set) if ref_set else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return precision, recall, f1


rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)


def calculate_metrics(references, predictions, latencies, sources):
    rouge_p, rouge_r, rouge_f = [], [], []
    target_p, target_r, target_f = [], [], []
    source_p = []

    for ref, pred, source in zip(references, predictions, sources):
        rouge_result = rouge.score(str(ref), str(pred))["rougeL"]

        rouge_p.append(rouge_result.precision)
        rouge_r.append(rouge_result.recall)
        rouge_f.append(rouge_result.fmeasure)

        p, r, f = lexical_overlap(ref, pred)

        target_p.append(p)
        target_r.append(r)
        target_f.append(f)

        sp, _, _ = lexical_overlap(source, pred)
        source_p.append(sp)

    total_latency = float(np.sum(latencies))

    return {
        "samples": len(predictions),

        "rougeL_precision": float(np.mean(rouge_p)),
        "rougeL_recall": float(np.mean(rouge_r)),
        "rougeL_f1": float(np.mean(rouge_f)),

        "target_lexical_precision": float(np.mean(target_p)),
        "target_lexical_recall": float(np.mean(target_r)),
        "target_lexical_f1": float(np.mean(target_f)),

        "source_lexical_precision": float(np.mean(source_p)),
        "hallucination_risk_pct": float(
            (1 - np.mean(source_p)) * 100
        ),

        "mean_latency_seconds": float(np.mean(latencies)),
        "total_latency_seconds": total_latency,
        "throughput_samples_per_second": (
            len(predictions) / total_latency
            if total_latency > 0 else 0.0
        )
    }

print("Dataset and metric engine ready.")

In [ ]:
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")

login(token=HF_TOKEN)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True
)

model.eval()

print("Llama 3.1 8B loaded successfully.")

In [ ]:
import gc

MAX_EVAL_SAMPLES = 20
MAX_TEST_SAMPLES = 20

def generate_highlight(abstract):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert scientific editor. "
                "Generate a concise, accurate scientific highlight "
                "based only on the supplied abstract. "
                "Do not introduce unsupported information. "
                "Return only the highlight."
            )
        },
        {
            "role": "user",
            "content": str(abstract)
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    latency = time.time() - start_time

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return prediction, latency


# Validation evaluation
eval_df = val_df.copy()

if MAX_EVAL_SAMPLES is not None:
    eval_df = eval_df.head(MAX_EVAL_SAMPLES)

val_predictions = []
val_latencies = []

for i, row in enumerate(eval_df.itertuples(index=False), 1):
    prediction, latency = generate_highlight(row.Abstract)

    val_predictions.append(prediction)
    val_latencies.append(latency)

    print(f"Validation {i}/{len(eval_df)} | {latency:.2f}s")


metrics = calculate_metrics(
    references=eval_df["Highlights"].tolist(),
    predictions=val_predictions,
    latencies=val_latencies,
    sources=eval_df["Abstract"].tolist()
)

benchmark_result = {
    "model": "Llama-3.1-8B-Instruct",
    "fine_tuning": False,
    "quantization": "4-bit NF4",
    "evaluation_type": "zero-shot",
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "metrics": metrics
}

benchmark_path = os.path.join(
    RESULTS_DIR,
    "benchmark_llama-3.1-8b.json"
)

with open(benchmark_path, "w") as f:
    json.dump(benchmark_result, f, indent=4)


validation_output = eval_df[["Abstract", "Highlights"]].copy()
validation_output["Generated_Highlight"] = val_predictions
validation_output["Latency_Seconds"] = val_latencies

validation_output.to_csv(
    os.path.join(RESULTS_DIR, "llama_validation_predictions.csv"),
    index=False
)

print("\nVALIDATION RESULTS")
print(json.dumps(metrics, indent=4))


# Masked test predictions
if test_df is not None:
    test_eval_df = test_df.copy()

    if MAX_TEST_SAMPLES is not None:
        test_eval_df = test_eval_df.head(MAX_TEST_SAMPLES)

    test_predictions = []

    for i, row in enumerate(test_eval_df.itertuples(index=False), 1):
        prediction, latency = generate_highlight(row.Abstract)
        test_predictions.append(prediction)

        print(f"Test {i}/{len(test_eval_df)} | {latency:.2f}s")

    test_output = test_eval_df[["Abstract"]].copy()
    test_output["Generated_Highlight"] = test_predictions

    test_output.to_csv(
        os.path.join(
            RESULTS_DIR,
            "MixSub-SciHigh_test_predictions_llama-3.1-8b.csv"
        ),
        index=False
    )

    print("Test predictions saved.")
else:
    print("Masked test file unavailable or missing Abstract column. Skipping.")

print("\nAll available outputs saved to:", RESULTS_DIR)

del model
gc.collect()
torch.cuda.empty_cache()